In [25]:
import Helper as req
import json
import pandas as pd
from pandas import DataFrame
import requests


In [26]:

# GLOBAL VARIABLES

matches_file = "matches.json"
data_file = "data.pkl"
results_file = "results.json"
api_key = "RGAPI-02033ac0-325a-4688-bdef-96bfdcf64cc9"
riot_id = "Ferix8475#NA1"

gameName = riot_id.split("#")[0]
tagline = riot_id.split("#")[1]

puuid = req.fetch_account_puuid(gameName, tagline, api_key)


# UPDATE THE DATAFRAME
req.update_data(puuid=puuid, api_key=api_key)

df = pd.read_pickle(data_file)

15
New Match, NA1_5038189568
New Match, NA1_5038178464
New Match, NA1_5038159577
New Match, NA1_5038123590
New Match, NA1_5037562725
New Match, NA1_5037548204
New Match, NA1_5037523524
New Match, NA1_5037029025
New Match, NA1_5037014505
New Match, NA1_5036999177
New Match, NA1_5036982297
New Match, NA1_5036948458
New Match, NA1_5036930344
New Match, NA1_5036865546
New Match, NA1_5036853759


In [27]:
df.to_csv("output.csv")
df


,Champion,Role,Patch,Win,Summoner1,Summoner2,Turrets_Killed,Total_Minions_Killed,Total_Jungle_Monsters_Killed,Total_Damage_DealtToChampions,...,Flex_Rune,Offense_Rune,Primary_Tree,Primary_Keystone,Primary_Choice1,Primary_Choice2,Primary_Choice3,Secondary_Tree,Secondary_Choice1,Secondary_Choice2
0,Zed,JUNGLE,14.13,True,11,4,1,21,173,23043,...,5008,5005,8100,8112,8143,8138,8106,8000,9105,8014
0,Sett,TOP,14.13,False,12,4,3,291,2,58572,...,5008,5005,8000,8010,9111,9105,8299,8400,8473,8451
0,LeeSin,JUNGLE,14.13,True,11,4,3,33,148,25527,...,5008,5005,8000,8010,9111,9104,8014,8300,8304,8347
0,Zed,MIDDLE,14.13,True,12,4,3,217,32,27508,...,5008,5008,8300,8369,8304,8313,8347,8200,8210,8237
0,Lux,MIDDLE,14.13,True,12,4,7,221,0,33501,...,5008,5008,8200,8229,8226,8210,8237,8300,8321,8347
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,Zed,MIDDLE,14.13,True,12,4,5,301,8,81264,...,5008,5008,8300,8369,8304,8313,8347,8200,8210,8237
0,Lux,UTILITY,14.13,True,14,4,3,57,2,36426,...,5008,5008,8200,8229,8226,8210,8237,8100,8126,8106
0,Jayce,JUNGLE,14.13,False,11,4,0,45,150,13672,...,5008,5005,8300,8369,8304,8313,8347,8100,8105,8143
0,Darius,TOP,14.13,True,6,4,5,258,8,58363,...,5008,5005,8000,8010,9111,9105,8299,8400,8444,8453


In [28]:
def purge_df(df : DataFrame):
    df = df[~df.isin(['']).any(axis=1)]
    df = df.dropna()
    return df

In [29]:
objective_df = df.groupby(['Champion', 'Role', 'Win']).agg({
    'Barons_Killed': 'mean',
    'Void_Grubs_Killed': 'mean',
    'Dragons_Killed': 'mean',
    'Turrets_Killed': 'mean',
    'Rift_Heralds_Killed': 'mean',

}).reset_index()
objective_df = purge_df(objective_df)
objective_df

,Champion,Role,Win,Barons_Killed,Void_Grubs_Killed,Dragons_Killed,Turrets_Killed,Rift_Heralds_Killed
2,Aatrox,JUNGLE,False,0.000000,0.666667,0.666667,0.666667,0.666667
3,Aatrox,JUNGLE,True,0.500000,2.5,2.500000,1.000000,0.000000
4,Aatrox,MIDDLE,False,0.000000,0.0,0.000000,0.500000,0.000000
5,Aatrox,MIDDLE,True,0.500000,1.0,3.500000,4.000000,0.500000
6,Aatrox,TOP,False,0.214286,0.928571,1.285714,1.285714,0.500000
...,...,...,...,...,...,...,...,...
172,Zed,MIDDLE,True,0.588235,2.034483,2.088235,2.647059,0.617647
173,Zed,TOP,False,0.000000,0.0,1.000000,1.000000,0.000000
174,Zed,TOP,True,1.000000,4.5,2.250000,4.000000,0.750000
175,Zed,UTILITY,False,0.000000,0.0,1.000000,0.000000,1.000000


In [30]:
grouped = df.groupby(['Champion', 'Role', 'Win']).size().unstack(fill_value=0)
grouped['Winrate'] = grouped[True] / (grouped[True] + grouped[False]) * 100
grouped['Games Played'] = (grouped[True] + grouped[False]) 
general_df = grouped[['Winrate', 'Games Played']].reset_index()
general_df = purge_df(general_df)
general_df

Win,Champion,Role,Winrate,Games Played
1,Aatrox,JUNGLE,40.000000,5
2,Aatrox,MIDDLE,50.000000,4
3,Aatrox,TOP,44.000000,25
4,Aatrox,UTILITY,30.769231,13
5,Ahri,MIDDLE,50.000000,2
...,...,...,...,...
116,Zac,UTILITY,33.333333,3
118,Zed,JUNGLE,55.932203,59
119,Zed,MIDDLE,52.307692,65
120,Zed,TOP,80.000000,5


In [31]:
effectiveness_df = df.groupby(['Champion', 'Role', 'Win']).agg({
        'Total_Minions_Killed': 'mean',
        'Total_Jungle_Monsters_Killed': 'mean',
        'Total_Damage_DealtToChampions': 'mean',
        'KDA': 'mean',
        'Kill_Participation': 'mean',
        'Damage_Share': 'mean',
        'Turret_Plates_Taken': 'mean',
        'Gold_Per_Minute': 'mean',
        'Damage_Per_Minute': 'mean',
        'Vision_Score_Per_Minute': 'mean',
        'Lane_Minions_Before_10_Minutes': 'mean',
        'Jungle_CS_Before_10_Minutes': 'mean',
        'Sol_Kills':'mean'

}).reset_index()
effectiveness_df = purge_df(effectiveness_df)

effectiveness_df


,Champion,Role,Win,Total_Minions_Killed,Total_Jungle_Monsters_Killed,Total_Damage_DealtToChampions,KDA,Kill_Participation,Damage_Share,Turret_Plates_Taken,Gold_Per_Minute,Damage_Per_Minute,Vision_Score_Per_Minute,Lane_Minions_Before_10_Minutes,Jungle_CS_Before_10_Minutes,Sol_Kills
2,Aatrox,JUNGLE,False,21.666667,117.666667,7802.333333,1.722222,0.344144,0.108886,0.666667,360.084205,270.202989,0.835186,0.0,52.283333,0.0
3,Aatrox,JUNGLE,True,32.000000,143.000000,17619.500000,3.166667,0.357558,0.162618,0.0,391.677828,474.186637,0.823045,2.0,56.500000,1.5
4,Aatrox,MIDDLE,False,116.000000,0.000000,12644.000000,0.902778,0.291379,0.214181,0.5,385.095136,561.395316,0.643855,51.5,0.000000,1.0
5,Aatrox,MIDDLE,True,203.500000,15.000000,40967.500000,3.811111,0.479167,0.237691,1.5,469.629052,1051.767274,0.868906,59.5,0.000000,1.5
6,Aatrox,TOP,False,177.142857,7.000000,21756.214286,1.387915,0.337247,0.225665,1.357143,358.042213,650.800007,0.644207,56.285714,0.000000,2.142857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,Zed,MIDDLE,True,198.794118,7.970588,29206.941176,4.525175,0.438761,0.252082,2.558824,476.529360,872.486034,0.727264,66.058824,0.094118,4.147059
173,Zed,TOP,False,211.000000,5.000000,21874.000000,0.400000,0.181818,0.183768,1.0,320.356173,584.754435,0.835389,59.0,0.000000,2.0
174,Zed,TOP,True,251.750000,32.750000,64187.250000,6.262500,0.574474,0.358576,3.75,572.681762,1588.495508,0.996436,79.25,0.000000,8.25
175,Zed,UTILITY,False,53.000000,0.000000,20131.000000,1.428571,0.454545,0.248205,1.0,331.607087,757.125193,2.218990,14.0,0.000000,2.0


In [32]:
winrate_by_role = df.groupby(['Champion', 'Role']).agg(
    Winrate=('Win', 'mean'),
    Games_Played=('Win', 'count')
).reset_index()
winrate_by_role = purge_df(winrate_by_role)
winrate_by_role

,Champion,Role,Winrate,Games_Played
1,Aatrox,JUNGLE,0.400000,5
2,Aatrox,MIDDLE,0.500000,4
3,Aatrox,TOP,0.440000,25
4,Aatrox,UTILITY,0.307692,13
5,Ahri,MIDDLE,0.500000,2
...,...,...,...,...
116,Zac,UTILITY,0.333333,3
118,Zed,JUNGLE,0.559322,59
119,Zed,MIDDLE,0.523077,65
120,Zed,TOP,0.800000,5


In [33]:
# Calculate Winrates by Item and Champion
melted_df = df.melt(id_vars=['Champion', 'Win'], value_vars=['Item0', 'Item1', 'Item2', 'Item3', 'Item4', 'Item5', 'Item6'],
                    var_name='ItemSlot', value_name='Item')

melted_df = melted_df[melted_df['Item'] != 0] # Get rid of empty items

item_winrate_df = melted_df.groupby(['Champion', 'Item']).agg(
    Winrate=('Win', 'mean'),
    Games_Played=('Win', 'count')
).reset_index()

mydict = req.json_extract_important_items()

items_to_keep = list(mydict.keys())
item_winrate_df = item_winrate_df[item_winrate_df['Item'].isin(items_to_keep)] # Get rid of unwanted items (components, epic items)

item_winrate_df['Item'] = item_winrate_df['Item'].replace(mydict) # Replace IDs with Names
item_winrate_df = purge_df(item_winrate_df)
item_winrate_df

,Champion,Item,Winrate,Games_Played
6,Aatrox,Doran's Shield,0.222222,9
7,Aatrox,Doran's Blade,0.500000,4
13,Aatrox,Control Ward,0.333333,3
14,Aatrox,Evenshroud,0.000000,1
15,Aatrox,Guardian Angel,1.000000,2
...,...,...,...,...
968,Zed,Axiom Arc,0.823529,17
969,Zed,Hubris,0.818182,11
970,Zed,Profane Hydra,0.636364,55
971,Zed,Voltaic Cyclosword,0.653061,49


In [34]:
runes = req.json_extract_runes()
treepage = {
    8000: "Precision",
    8100: "Domination",
    8200: "Sorcery",
    8300: "Inspiration",
    8400: "Resolve"
}


df['Primary_Tree'] = df['Primary_Tree'].replace(treepage)
df['Secondary_Tree'] = df['Secondary_Tree'].replace(treepage)
df['Primary_Keystone'] = df['Primary_Keystone'].replace(runes)
runes

{8369: 'First Strike',
 8446: 'Demolish',
 8126: 'Cheap Shot',
 8415: 'The Arcane Colossus',
 8410: 'Approach Velocity',
 8232: 'Waterwalking',
 8299: 'Last Stand',
 8112: 'Electrocute',
 8234: 'Celerity',
 8453: 'Revitalize',
 8360: 'Unsealed Spellbook',
 8004: 'The Brazen Perfect',
 8128: 'Dark Harvest',
 8220: 'The Calamity',
 8016: 'The Merciless Elite',
 8473: 'Bone Plating',
 8339: 'Celestial Body',
 8214: 'Summon Aery',
 8237: 'Scorch',
 8139: 'Taste of Blood',
 9101: 'Absorb Life',
 8008: 'Lethal Tempo',
 8010: 'Conqueror',
 9105: 'Legend: Haste',
 8106: 'Ultimate Hunter',
 8017: 'Cut Down',
 8224: 'Nullifying Orb',
 8210: 'Transcendence',
 8005: 'Press the Attack',
 8435: 'Mirror Shell',
 8115: 'The Aether Blade',
 8359: 'Kleptomancy',
 8352: 'Time Warp Tonic',
 5003: 'Magic Resist',
 8135: 'Treasure Hunter',
 8120: 'Ghost Poro',
 8134: 'Ingenious Hunter',
 8351: 'Glacial Augment',
 8242: 'Unflinching',
 8401: 'Shield Bash',
 9111: 'Triumph',
 8105: 'Relentless Hunter',
 8454:

In [35]:
moregrouped = df.groupby(['Champion', 'Role', 'Primary_Tree', 'Secondary_Tree']).agg(
    Winrate=('Win', 'mean'),
    Games_Played=('Win', 'count')
).reset_index()
moregrouped = purge_df(moregrouped)
moregrouped

,Champion,Role,Primary_Tree,Secondary_Tree,Winrate,Games_Played
1,Aatrox,JUNGLE,Precision,Domination,0.400000,5
2,Aatrox,MIDDLE,Precision,Resolve,0.500000,4
3,Aatrox,TOP,Precision,Domination,0.000000,1
4,Aatrox,TOP,Precision,Resolve,0.458333,24
5,Aatrox,UTILITY,Inspiration,Resolve,0.000000,1
...,...,...,...,...,...,...
180,Zed,MIDDLE,Precision,Sorcery,0.500000,2
181,Zed,TOP,Domination,Inspiration,0.000000,1
182,Zed,TOP,Domination,Sorcery,1.000000,3
183,Zed,TOP,Precision,Domination,1.000000,1


In [36]:
zed_info = moregrouped[moregrouped['Champion'] == 'Zed']
zed_info

,Champion,Role,Primary_Tree,Secondary_Tree,Winrate,Games_Played
165,Zed,JUNGLE,Domination,Precision,0.500000,4
166,Zed,JUNGLE,Domination,Sorcery,0.846154,13
167,Zed,JUNGLE,Inspiration,Domination,0.470588,17
168,Zed,JUNGLE,Inspiration,Precision,0.500000,12
169,Zed,JUNGLE,Inspiration,Sorcery,0.500000,4
170,Zed,JUNGLE,Precision,Domination,0.800000,5
171,Zed,JUNGLE,Precision,Sorcery,0.000000,4
172,Zed,MIDDLE,Domination,Inspiration,0.500000,8
173,Zed,MIDDLE,Domination,Resolve,0.833333,6
174,Zed,MIDDLE,Domination,Sorcery,0.590909,22


In [37]:
keystone_runes_df = df.groupby(['Champion', 'Role', 'Primary_Keystone', 'Secondary_Tree']).agg(
    Winrate=('Win', 'mean'),
    Games_Played=('Win', 'count')
).reset_index()

keystone_runes_df = purge_df(keystone_runes_df)
keystone_runes_df

,Champion,Role,Primary_Keystone,Secondary_Tree,Winrate,Games_Played
1,Aatrox,JUNGLE,Conqueror,Domination,0.400000,5
2,Aatrox,MIDDLE,Conqueror,Resolve,0.500000,4
3,Aatrox,TOP,Conqueror,Domination,0.000000,1
4,Aatrox,TOP,Conqueror,Resolve,0.458333,24
5,Aatrox,UTILITY,Arcane Comet,Domination,0.000000,1
...,...,...,...,...,...,...
183,Zed,MIDDLE,First Strike,Sorcery,0.500000,14
184,Zed,TOP,Conqueror,Domination,1.000000,1
185,Zed,TOP,Electrocute,Inspiration,0.000000,1
186,Zed,TOP,Electrocute,Sorcery,1.000000,3


In [38]:
print(req.json_extract_runes())
treepage = {
    8000: "Precision",
    8100: "Domination",
    8200: "Sorcery",
    8300: "Inspiration",
    8400: "Resolve"
}

{8369: 'First Strike', 8446: 'Demolish', 8126: 'Cheap Shot', 8415: 'The Arcane Colossus', 8410: 'Approach Velocity', 8232: 'Waterwalking', 8299: 'Last Stand', 8112: 'Electrocute', 8234: 'Celerity', 8453: 'Revitalize', 8360: 'Unsealed Spellbook', 8004: 'The Brazen Perfect', 8128: 'Dark Harvest', 8220: 'The Calamity', 8016: 'The Merciless Elite', 8473: 'Bone Plating', 8339: 'Celestial Body', 8214: 'Summon Aery', 8237: 'Scorch', 8139: 'Taste of Blood', 9101: 'Absorb Life', 8008: 'Lethal Tempo', 8010: 'Conqueror', 9105: 'Legend: Haste', 8106: 'Ultimate Hunter', 8017: 'Cut Down', 8224: 'Nullifying Orb', 8210: 'Transcendence', 8005: 'Press the Attack', 8435: 'Mirror Shell', 8115: 'The Aether Blade', 8359: 'Kleptomancy', 8352: 'Time Warp Tonic', 5003: 'Magic Resist', 8135: 'Treasure Hunter', 8120: 'Ghost Poro', 8134: 'Ingenious Hunter', 8351: 'Glacial Augment', 8242: 'Unflinching', 8401: 'Shield Bash', 9111: 'Triumph', 8105: 'Relentless Hunter', 8454: 'The Leviathan', 8275: 'Nimbus Cloak', 82